In [1]:
import json
import logging
import os
import re
from typing import List, Union

import datasets
import hydra
import pandas
import phonemizer
from phonemizer.backend import EspeakBackend
import soundfile
from omegaconf import DictConfig
from torch.utils.data import Dataset

logger: logging.Logger = logging.getLogger(__name__)

/home/malo/.pyenv/versions/vibravox/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, Audio, DatasetDict 
COMMON_VOICE = "mozilla-foundation/common_voice_13_0"

dataset_name: str = COMMON_VOICE
language: str = "fr"

path_to_processor_config = "configs/lightning_module/dnn_module/processor_config/wav2vec2processor" # relative path from './'

In [3]:
# French vowels
vowels = ["i", "e", "ɛ", "a", "ɑ", "o", "ɔ", "u", "y", "ø", "œ", "ə"]
# tilde
tilde = ["̃"]
# Semi-vowels
semi_vowels = ["j", "w", "ɥ"]
# French consonants
consonants = [
    "p",
    "b",
    "t",
    "d",
    "k",
    "ɡ",
    "f",
    "v",
    "s",
    "z",
    "ʃ",
    "ʒ",
    "m",
    "n",
    "ɲ",
    "ŋ",
    "l",
    "ʁ",
]
# Other symbols
other_symbols = [" "]
# Complete list
french_phonetic_alphabet = vowels + tilde + semi_vowels + consonants + other_symbols

In [4]:
dataset_dict = load_dataset(dataset_name, language)

# keep only train, val and test
dataset_dict = DatasetDict(
    {
        "train": dataset_dict["train"],
        "validation": dataset_dict["validation"],
        "test": dataset_dict["test"],
    }
)

# ld_train = load_dataset(
#     dataset_name, language, split="train"
# )

# ld_eval = load_dataset(
#     dataset_name, language, split="validation"
# )

# ld_test = load_dataset(
#     dataset_name, language, split="test"
# )

In [5]:
def remove_unwanted_characters(
    batch: Union[pandas.DataFrame, Dataset], task_type: str = "phoneme"
) -> Union[pandas.DataFrame, Dataset]:
    """
    Removes special characters and transforms diacritics and ligatures into characters from the latin alphabet.

    Args:
        batch (`Union[pandas.DataFrame, Dataset]`): The dataset.
        task_type (str): Defaults to `phoneme`. Whether to do Speech-to-Text or Speech-to-Phoneme.
    Returns:
        `Union[pandas.DataFrame, Dataset]`: batch without special characters, diacritics and ligatures.
    """
    chars_to_remove_regex = '[\,\?\.\!\;\:"\(\)\#\d+]'

    # remove special characters
    try:
        batch["text"] = re.sub(chars_to_remove_regex, "", batch["text"]).lower().strip()
    except:
        try:
            batch["text"] = (
                re.sub(chars_to_remove_regex, "", batch["sentence"]).lower().strip()
            )
        except:
            batch["text"] = (
                re.sub(chars_to_remove_regex, "", batch["transcription"])
                .lower()
                .strip()
            )

    batch["text"] = re.sub("-", " ", batch["text"])
    batch["text"] = re.sub("_", " ", batch["text"])

    # remove annoying characters
    # diacritics
    if task_type == "text":
        batch["text"] = re.sub("â", "a", batch["text"])
        batch["text"] = re.sub("ā", "a", batch["text"])
        batch["text"] = re.sub("ä", "a", batch["text"])
        batch["text"] = re.sub("á", "a", batch["text"])
        batch["text"] = re.sub("ȧ", "a", batch["text"])
        batch["text"] = re.sub("à", "a", batch["text"])
        batch["text"] = re.sub("ã", "a", batch["text"])

        # batch["text"] = re.sub("é", "e", batch["text"]) # we keep 'é' because it has pronunciation usefulness
        # batch["text"] = re.sub("è", "e", batch["text"]) # we keep 'è' because it has pronunciation usefulness
        # batch["text"] = re.sub("ê", "e", batch["text"]) # we keep 'ê' because it has pronunciation usefulness
        batch["text"] = re.sub("ë", "e", batch["text"])
        batch["text"] = re.sub("ę", "e", batch["text"])

        batch["text"] = re.sub("ï", "i", batch["text"])
        batch["text"] = re.sub("î", "i", batch["text"])
        batch["text"] = re.sub("i", "i", batch["text"])  # i point suscrit
        batch["text"] = re.sub("í", "i", batch["text"])
        batch["text"] = re.sub("ī", "i", batch["text"])

        batch["text"] = re.sub("j", "j", batch["text"])  # j point suscrit

        batch["text"] = re.sub("ł", "l", batch["text"])

        batch["text"] = re.sub("ô", "o", batch["text"])
        batch["text"] = re.sub("ó", "o", batch["text"])
        batch["text"] = re.sub("ø", "o", batch["text"])
        batch["text"] = re.sub("ō", "o", batch["text"])
        batch["text"] = re.sub("õ", "o", batch["text"])
        batch["text"] = re.sub("ö", "o", batch["text"])

        batch["text"] = re.sub("š", "s", batch["text"])
        batch["text"] = re.sub("ṣ", "s", batch["text"])

        batch["text"] = re.sub("ṭ", "t", batch["text"])

        batch["text"] = re.sub("ù", "u", batch["text"])
        batch["text"] = re.sub("û", "u", batch["text"])
        batch["text"] = re.sub("ü", "u", batch["text"])
        batch["text"] = re.sub("ú", "u", batch["text"])

        batch["text"] = re.sub("ÿ", "y", batch["text"])
        batch["text"] = re.sub("y̌", "y", batch["text"])

        # batch["text"] = re.sub("ç", "c", batch["text"]) # we keep 'ç' because it has pronunciation usefulness
        batch["text"] = re.sub("č", "c", batch["text"])

        batch["text"] = re.sub("ñ", "n", batch["text"])
        batch["text"] = re.sub("ṇ", "n", batch["text"])

        batch["text"] = re.sub("ẓ", "z", batch["text"])
        batch["text"] = re.sub("ž", "z", batch["text"])

        # ligatures
        batch["text"] = re.sub("æ", "ae", batch["text"])
        batch["text"] = re.sub("œ", "oe", batch["text"])

        batch["text"] = re.sub("ꜳ", "aa", batch["text"])
        batch["text"] = re.sub("ꜵ", "ao", batch["text"])
        batch["text"] = re.sub("ꜷ", "au", batch["text"])
        batch["text"] = re.sub("ꜹ", "av", batch["text"])
        batch["text"] = re.sub("ꜽ", "ay", batch["text"])
        batch["text"] = re.sub("ȸ", "db", batch["text"])
        batch["text"] = re.sub("ʣ", "dz", batch["text"])
        batch["text"] = re.sub("ﬀ", "ff ", batch["text"])
        batch["text"] = re.sub("ﬁ", "fi", batch["text"])
        batch["text"] = re.sub("ﬂ", "fl", batch["text"])
        batch["text"] = re.sub("ﬃ", "ffi", batch["text"])
        batch["text"] = re.sub("ﬄ", "ffl", batch["text"])
        batch["text"] = re.sub("ĳ", "ij", batch["text"])
        batch["text"] = re.sub("ǉ", "lj", batch["text"])
        batch["text"] = re.sub("ǌ", "nj", batch["text"])
        batch["text"] = re.sub("ꝏ", "oo", batch["text"])
        batch["text"] = re.sub("ȹ", "qp", batch["text"])
        batch["text"] = re.sub("ﬆ", "st", batch["text"])
        batch["text"] = re.sub("ﬅ", "ft", batch["text"])
        batch["text"] = re.sub("ʦ", "ts", batch["text"])
        batch["text"] = re.sub("ᵫ", "ue", batch["text"])
        batch["text"] = re.sub("ꭣ", "uo", batch["text"])
        batch["text"] = re.sub("ꝡ", "vy", batch["text"])

    return batch

<>:13: SyntaxWarning: invalid escape sequence '\,'
<>:13: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_6610/2931045936.py:13: SyntaxWarning: invalid escape sequence '\,'
  chars_to_remove_regex = '[\,\?\.\!\;\:"\(\)\#\d+]'


In [6]:
f_remove_unwanted_characters = lambda batch: remove_unwanted_characters(
    batch, "phoneme"
)

dataset_dict = dataset_dict.map(f_remove_unwanted_characters)

In [7]:
def filter_dict_words(batch, list_str):
    for x in list_str:
        return not x in batch

In [8]:
list_unwanted_words = [ 
    'chapitre',
    'enregistr',
    'domaine',
    'public',
    'conte',
    'librivox',
    'org'
]

f_filter_dict_words = lambda batch: filter_dict_words(
    batch, list_unwanted_words
)

dataset_dict = dataset_dict.filter(f_filter_dict_words, input_columns=["text"])

In [9]:
def phonemize_characters(
    batch: Union[pandas.DataFrame, Dataset]
) -> Union[pandas.DataFrame, Dataset]:
    """
    Phonemizes text i.e. translates french strings to international phoneme alphabet french strings
    e.g. "Enchanté, je m'appelle Malo" -> "ãʃãte ʒə mapɛl malo"

    Args:
        batch (`Union[pandas.DataFrame, Dataset]`): The dataset.
    Returns:
        `Union[pandas.DataFrame, Dataset]`: The dataset but instead of french texts, it is phonemized french text.
    """
    backend = phonemizer.backend.EspeakBackend(
        language="fr-fr", language_switch="remove-utterance"
    )
    batch["phoneme"] = backend.phonemize(batch["text"], strip=True, njobs=1)

    return batch

In [10]:
logger.info("Phonemizing texts")
dataset_dict = dataset_dict.map(phonemize_characters, batched=True, batch_size=1_000)

logger.info("Removing sentences with bad phonemes")
filter_for_french_phonetic_alphabet = lambda x: all(
    char in french_phonetic_alphabet for char in x
)
dataset_dict = dataset_dict.filter(
    filter_for_french_phonetic_alphabet, input_columns=["phoneme"]
)

logger.info(
    "Filtering labels by length (keeping only phonemized labels with more than 4 phonemes in order to avoid NaNs in CTC loss)"
)

filter_phoneme_labels_by_length = lambda x: len(x) > 4
dataset_dict = dataset_dict.filter(
    filter_phoneme_labels_by_length, input_columns=["phoneme"]
)

In [11]:
def save_vocab(
    dataset: Union[pandas.DataFrame, Dataset],
    task_type: str = "phoneme",
) -> None:
    """
    Saves the processed vocab file as 'vocab.json', to be ingested by tokenizer.

    Args:
        dataset (`Union[pandas.DataFrame, Dataset]`): The dataset.
        processor (str): path to processor config
        task_type (str): Defaults to `phoneme`. Whether to do Speech-to-Text or Speech-to-Phoneme.
    Returns:
        NoneType: None
    """
    for split in ['train', 'validation', 'test']:
        vocab = construct_vocab(dataset[split][task_type])

        print(f"dataset['text'] = {dataset[split]['text'][:10]}")
        if task_type == "phoneme":
            print(f"dataset['phoneme'] = {dataset[split]['phoneme'][:10]}")

    vocab_dict = {v: k for k, v in enumerate(sorted(vocab))}
    vocab_dict["|"] = vocab_dict[" "]
    _ = vocab_dict.pop(" ")
    vocab_dict["[UNK]"] = len(vocab_dict)
    vocab_dict["[PAD]"] = len(vocab_dict)

    with open(f"./{path_to_processor_config}/vocab.json", "w", encoding="utf-8") as fl:
        json.dump(vocab_dict, fl, ensure_ascii=False)

    logger.info("Created Vocab file!")

def construct_vocab(texts: str) -> List[str]:
    """
    Get unique characters from all the text in a list.

    Args:
        texts (str): The texts.
    Returns:
        List[str]: a list of texts.
    """
    all_text = " ".join(texts)
    vocab = list(set(all_text))
    return vocab


def add_column_input_length(
    batch: Union[pandas.DataFrame, Dataset]
) -> Union[pandas.DataFrame, Dataset]:
    """
    Adds a column named 'input_length' in batch.

    Args:
        batch (`Union[pandas.DataFrame, Dataset]`): The dataset on which the function adds a new column.
    Returns:
        `Union[pandas.DataFrame, Dataset]`: batch but with the new added column 'input_length' in seconds
    """
    try:
        batch["input_length"] = soundfile.info(batch["file"]).duration
    except:
        batch["input_length"] = soundfile.info(batch["path"]).duration
    return batch

In [12]:
# Construct and save the vocab file
save_vocab(
    dataset_dict,
    task_type='phoneme'
)

dataset['text'] = ['trois semaines plus tard il mène campagne en afrique', "la rue tire son nom d'un ancien propriétaire du quartier ahuntsic", 'le larzac est un turboréacteur double corps double flux dépourvu de postcombustion', 'je demande à voir le boulet', "il fut relevé au par les sires d'aumont", 'elle est basée à heraklion en grèce', 'la commune comprend les quartiers de gräben rottstock et dahlen', 'chaque président est nommé pour un mandat de deux ans', 'normand brie à québec', 'il devient rapidement le chef de file des catholiques']
dataset['phoneme'] = ['tʁwa səmɛn ply taʁ il mɛn kɑ̃paɲ ɑ̃n afʁik', 'la ʁy tiʁ sɔ̃ nɔ̃ dœ̃n ɑ̃sjɛ̃ pʁɔpʁietɛʁ dy kaʁtje aœ̃tsik', 'lə laʁzak ɛt œ̃ tyʁboʁeaktœʁ dubl kɔʁ dubl fly depuʁvy də postkɔ̃bystjɔ̃', 'ʒə dəmɑ̃d a vwaʁ lə bulɛ', 'il fy ʁəlve o paʁ le siʁ domɔ̃', 'ɛl ɛ baze a əʁakliɔ̃ ɑ̃ ɡʁɛs', 'la kɔmyn kɔ̃pʁɑ̃ le kaʁtje də ɡʁabɛn ʁɔtstɔk e dalɛn', 'ʃak pʁezidɑ̃ ɛ nɔme puʁ œ̃ mɑ̃da də døz ɑ̃', 'nɔʁmɑ̃ bʁi a kebɛk', 'il dəvjɛ̃ ʁapidmɑ̃ lə ʃɛf

In [13]:
df = dataset_dict['train'].to_pandas()
df = df[['text', "phoneme"]]

In [14]:
# display every row of the dataframe with no limit of row length
# pandas.set_option('display.max_colwidth', None)
# df

In [15]:
len(df[df['phoneme'].str.contains('ɥ')])

45640

In [16]:
# mots en ɛː comme mère, fête et maître
# df[df['text'].str.contains('mère')] # affiche seulement ɛ

In [17]:
dataset_dict = dataset_dict.remove_columns(["text"])

In [18]:
# push to hub!
dataset_dict.push_to_hub("Cnam-LMSSC/common_voice_13_french_phoneme", private=True)

Uploading the dataset shards: 100%|██████████| 2/2 [01:15<00:00, 37.72s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/Cnam-LMSSC/common_voice_13_french_phoneme/commit/bfd27d3d0639daa759db65e00845bad4fab3b8ea', commit_message='Upload dataset', commit_description='', oid='bfd27d3d0639daa759db65e00845bad4fab3b8ea', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Cnam-LMSSC/common_voice_13_french_phoneme', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Cnam-LMSSC/common_voice_13_french_phoneme'), pr_revision=None, pr_num=None)

In [19]:
from transformers import Wav2Vec2CTCTokenizer
path_to_processor_config = "configs/lightning_module/dnn_module/processor_config/wav2vec2processor" # relative path from './'

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=f"./{path_to_processor_config}/vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [20]:
# tokenizer.push_to_hub("Cnam-LMSSC/common-voice-13-wav2vec2-tokenizer", private=True)